# SlimServe — Phase 1, Week 1: FP16 baseline

Serve `Qwen2.5-7B-Instruct` with vLLM on Kaggle's **2× T4** and record the FP16 baseline row of the hero benchmark table.

**Before running:**
1. Settings → **Accelerator = GPU T4 × 2**
2. Settings → **Internet = On** (needed for `pip` + `git clone` + model download)
3. Edit `REPO_URL` below to your GitHub repo.

This notebook is deliberately thin — all logic lives in the `slimserve` package (clean, testable, reused across every phase). The notebook just clones, installs, and runs.

In [ ]:
# Confirm we have two T4s
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
import os

# HTTPS URL — Kaggle can't clone over SSH. If the repo is PRIVATE, use a token:
#   REPO_URL = f"https://{os.environ['GH_TOKEN']}@github.com/tanishq0421/slimserve.git"
# and add GH_TOKEN as a Kaggle Secret (Add-ons -> Secrets).
REPO_URL = "https://github.com/tanishq0421/slimserve.git"

if not os.path.exists("slimserve"):
    !git clone {REPO_URL}
%cd slimserve
!pip install -q -e .

## 2. Install vLLM

First run takes a few minutes (vLLM pulls a compatible torch + transformers).

In [ ]:
!pip install -q vllm

## 3. Configure the engine

`tensor_parallel_size=2` spans **both** T4s so the 7B fits comfortably in FP16 (a single 16GB T4 would OOM under load).

In [ ]:
import slimserve.pipeline  # side-effect: registers all implementations
from slimserve.core.config import EngineConfig, Precision
from slimserve.core.registry import build

REF_GPU_USD = 0.40  # reference price for 2× T4 / hr; cite your source in the README

engine_cfg = EngineConfig(
    name="vllm",
    model_path="Qwen/Qwen2.5-7B-Instruct",
    precision=Precision.FP16,
    kv_cache_quant=False,
    max_num_seqs=256,
    extra={"tensor_parallel_size": 2},
)

# Load the model ONCE and reuse it for the sanity check + benchmark.
engine = build("engine", "vllm", engine_cfg)

## 4. Sanity check — does it emit a valid tool call?

In [ ]:
from slimserve.benchmark.workload import build_tool_calling_workload

sample = build_tool_calling_workload(1)[0]
print("PROMPT:", sample.prompt)
print("OUTPUT:", engine.generate(sample).text)

## 5. Run the benchmark → record the baseline row

In [ ]:
from slimserve.benchmark.runner import BenchmarkRunner
from slimserve.benchmark.results_store import ResultsStore

runner = BenchmarkRunner(gpu_hourly_usd=REF_GPU_USD)
result = runner.run(
    config_name="teacher_fp16",
    engine=engine,
    params_b=7.0,
    precision="fp16",
)
ResultsStore().append(result)
result

## 6. The hero table so far

In [ ]:
import pandas as pd

pd.read_csv("results/benchmarks.csv")

---
### Save your results
`results/benchmarks.csv` lives in the Kaggle session and is wiped on recycle. To keep it:
- **Download** it from the file browser, **or**
- commit + push from the notebook (needs a GitHub token as a Kaggle Secret).

### Next (Week 2)
- INT8 → INT4 (AWQ) rows + KV-cache quantization (each on a **single** T4).
- Wire the real **BFCL** evaluator to replace the Week-1 validity stand-in.
- Tune continuous batching (`max_num_seqs`) and add the true-TTFT online-server measurement.